In [ ]:
from datatools import get_price
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from scipy.signal import argrelextrema

In [ ]:
data = get_price(start_date='2013-12-31', end_date='2025-12-31', fq='post', fields=None, data_path='data/daily_K')
data = data.sort_values(['ts_code', 'trade_date'])

In [ ]:
# 获取长期历史数据
df = get_price('000300.XSHG', start_date='2005-01-01', end_date='2025-12-31', fields=['close'], fq='post')

# 找局部高点（order=60 表示前后60个交易日内最高）
# order值越大，高点越显著（越稀疏）；越小，高点越多（包含小波峰）
high_idx = argrelextrema(df['close'].values, np.greater_equal, order=60)[0]
highs = df.iloc[high_idx].copy()
highs['date'] = highs.index

# 计算相邻高点间隔
highs = highs.reset_index(drop=True)
highs['prev_date'] = highs['date'].shift(1)
highs['interval_days'] = (highs['date'] - highs['prev_date']).dt.days
highs['interval_years'] = highs['interval_days'] / 365.25

# 过滤掉第一次
highs = highs.dropna()

# 统计
mean_interval = highs['interval_years'].mean()
median_interval = highs['interval_years'].median()

print("=== 显著高点验证结果（局部峰值，非历史新高） ===")
print(f"总共识别出 {len(highs)} 个显著高点")
print(f"平均间隔：{mean_interval:.2f} 年")
print(f"中位数间隔：{median_interval:.2f} 年")
print(f"间隔标准差：{highs['interval_years'].std():.2f} 年")
print("\n各次高点间隔（年）：")
print(highs[['date', 'close', 'interval_years']].round(2))

# 可视化
plt.figure(figsize=(14, 8))
plt.plot(df.index, df['close'], label='沪深300指数', linewidth=1.5)
plt.scatter(highs['date'], highs['close'], color='red', s=100, label='显著高点（order=60）')

# 画6年周期参考线
for i in range(len(highs)):
    plt.axvline(highs['date'].iloc[i], color='gray', linestyle='--', alpha=0.5)

plt.title('沪深300指数显著高点分布与6年周期验证')
plt.xlabel('日期')
plt.ylabel('指数点位')
plt.legend()
plt.grid(True)
plt.show()

# 间隔分布直方图
plt.figure(figsize=(10, 5))
plt.hist(highs['interval_years'], bins=10, alpha=0.7, color='skyblue', edgecolor='black')
plt.axvline(6, color='red', linestyle='--', label='假设6年周期')
plt.title('显著高点间隔分布直方图')
plt.xlabel('相邻高点间隔（年）')
plt.ylabel('出现次数')
plt.legend()
plt.grid(True)
plt.show()